In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv("../data/processed/cases_final.csv")

vectorizer = TfidfVectorizer(max_features=5000)
tfidf_matrix = vectorizer.fit_transform(df["text_full"])

df.head()

,case_id,file_name,text_full,no_perkara,penggugat,tergugat,amar_putusan,jenis_perkara,pengadilan
0,case_001,putusan_105_pdt.g_2025_pn_sda_20260624161353.pdf,penetapan nomor 105/pdt.g/2025/pn sda esi demi...,105/pdt.g/2025/pnsda,1. sri pudjiastuti bertempat tinggal di jalan....,1. siti rusmala bertempat tinggal di desa boha...,1. mengabulkan pencabutan permohonan perkara ...,Wanprestasi,PN Sidoarjo
1,case_002,putusan_148_pdt.g_2025_pn_sda_20260624161238.pdf,putusan nomor 148/pdt.g/2025/pn sda demi keadi...,148/pdt.g/2025/pnsda,-,les 1. kepala dinas perhubungan kabupaten sido...,nomor 415.4/34/438.5.13/2023nomor 107/lds-sdm...,Wanprestasi,PN Sidoarjo
2,case_003,putusan_149_pdt.g_2017_pn_sda_20260624160009.pdf,direktori putusan mahkamah agung republik indo...,149/pdt.g/2017/pnsda,onesi andry tejokusuma berkedudukan di jl. jem...,h. imam ghozali atau imam ghozali bertempat ti...,dalamkonpensi dalam eksepsi nkamah dalam poko...,Wanprestasi,PN Sidoarjo
3,case_004,putusan_166_pdt.g_2021_pn_sda_20260624160223.pdf,direktori putusan mahkamah agung pdt.i.c.3 put...,166/pdt.g/2021/pnsda,raditya afrisal hidayat berkedudukan di perum ...,rachmad harmoko bertempat tinggal di jalan kar...,1. menyatakan bahwa tergugat yang telah dipan...,Wanprestasi,PN Sidoarjo
4,case_005,putusan_213_pdt.g_2026_pn_sda_20260624160912.pdf,penetapan nomor 213/pdt.g/2026/pn sda demi kea...,213/pdt.g/2026/pnsda,wahjudi santoso tempat dan tanggal lahir sidoa...,avi anthy nurafni tempat dan tanggal lahir ung...,perkara ini setelah membaca penetapan hakim n...,Wanprestasi,PN Sidoarjo


In [3]:
def retrieve(query, k=5):
    query_vector = vectorizer.transform([query])
    similarity = cosine_similarity(query_vector, tfidf_matrix).flatten()
    top_index = similarity.argsort()[-k:][::-1]

    result = df.iloc[top_index].copy()
    result["similarity_score"] = similarity[top_index]

    return result

In [4]:
def predict_outcome(query):
    top_cases = retrieve(query, k=5)

    best_case = top_cases.iloc[0]

    return {
        "predicted_solution": best_case["amar_putusan"],
        "best_case": best_case["case_id"],
        "top_5_case_ids": ", ".join(top_cases["case_id"]),
        "similarity_score": best_case["similarity_score"]
    }

In [5]:
query = """
tergugat tidak memenuhi kewajiban pembayaran sesuai perjanjian
sehingga penggugat mengalami kerugian dan meminta ganti rugi
"""

hasil = predict_outcome(query)
hasil

{'predicted_solution': ' 1. menyatakan tergugat i dan tergugat ll yang telah dipanggil secara sah dan patut untuk menghadap di persidangan tidak hadir. 2. mengabulkan gugatan penggugat secara verstek untuk sebagian. 3. menyatakan sah dan mengikat dengan segala akibat hukumnya surat 27-1-2017. 4. menyatakan tergugat i dan tergugat ill telah ingkar janji/wanprestasi kepada penggugat. 5. menyatakan sah dan mengikat sebagian pembayaran hutang tergugat i dan ribu rupiah nes halatan15dari17putusannomor 390/pdt.g/2023/pnsda   6. menghukum tergugat i dan tergugat il untuk membayar kekurangan hutangnya sebesar rp. 166.500.000 - seratus enam puluh enam juta lima ratus ribu rupiah kepada penggugat. esi 7. menghukum tergugat i dan tergugat li untuk membayar biaya perkara sebesar rp. 5.110.000 00 lima juta seratus sepuluh ribu rupiah secara tanggung renteng. 8. menolak selain dan untuk selebihnya. demikianlah diputuskan dalam rapat permusyawaratan majelis hakim pengadilan negeri sidoarjo pada hari 

In [6]:
prediction_df = pd.DataFrame([{
    "query_id": "query_001",
    "query_text": query,
    "predicted_solution": hasil["predicted_solution"],
    "best_case": hasil["best_case"],
    "top_5_case_ids": hasil["top_5_case_ids"],
    "similarity_score": hasil["similarity_score"]
}])

prediction_df.to_csv("../data/results/predictions.csv", index=False)

prediction_df

,query_id,query_text,predicted_solution,best_case,top_5_case_ids,similarity_score
0,query_001,\ntergugat tidak memenuhi kewajiban pembayaran...,1. menyatakan tergugat i dan tergugat ll yang...,case_023,"case_023, case_024, case_018, case_017, case_022",0.268872
